# Figure 1 — what is recoverable

Figure 1 was built in a different project directory, by Yang Li (panels c, d, e,
g) and Quinn Hauck (panel h), and handed over as a code bundle at
`yang_lc2/leafcutter2_paper_figures/`. That bundle contains no data; the pipeline
outputs it consumes still live in Yang's directory, which this notebook reads
read-only.

| Panel | Status | Where it comes from |
|---|---|---|
| **a**, **b** | not code | schematics |
| **c** | **tracks only** | the panel is an IGV screenshot; the BED12 junction tracks are recovered |
| **d** | **rebuilt** | `junctions_{Q1..Q4,ALL}.tsv` from `parse_classification_claude.py` |
| **e** | **rebuilt** | same tables, GENCODE composition per class |
| **f** | elsewhere | the simulation benchmark, in [bfairkun/20260825_leaf2simulation_paper](https://github.com/bfairkun/20260825_leaf2simulation_paper) |
| **g** | **rebuilt** | `intron_log2fc.pickle`, the cache from `cl_plot_log2fc_dist.py` |
| **h** | **rebuilt** | the rule statistics recomputed from the pipeline outputs, plus Quinn's matched control set |
| **i** | **not recoverable** | nothing in either directory produces it |

Two notes on what "rebuilt" means for panel h. The script that rendered the
published heatmap was never handed over — `plot_rules.py` in the bundle is a
stale Feb-2025 layout with the numbers pasted in as literals — so the statistics
here are recomputed from the same inputs the two Rmds use, and the heatmap is
drawn fresh. And the productive column is a *matched control set* built by a fork
of the classifier: the main run's `leafcutter2_exon_stats.txt` covers only
unproductive junctions, so the control cannot come from it.

In [ ]:
import os
from matplotlib import pyplot as plt

import Figure1_helpers as H
import Figure1_plot_helpers as P

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

PLOTS_DIR = 'plots'
os.makedirs(PLOTS_DIR, exist_ok=True)
print('reading pipeline output from', H.SOURCE)

In [ ]:
# Fast path: reload what run_all() pickled.
data = H.load_plot_data('figure_data')

fig1c_tracks = data['fig1c_tracks']
fig1d_data   = data['fig1d_data']
fig1e_data   = data['fig1e_data']
fig1g_data   = data['fig1g_data']
fig1h_data   = data['fig1h_data']

In [ ]:
# Full rebuild. Reads the 50 MB classification table and the 43 MB log2FC
# cache, so it takes a few minutes. Run once, then use the cell above.

# data = H.run_all('figure_data')

### Fig. 1c — SRSF4 browser tracks

The published panel is an IGV screenshot, so there is no plotting code to
recover. What is recovered is the data behind the junction track: the BED12
records, joined to the LeafCutter2 class and GENCODE annotation of each junction,
restricted to *SRSF4*. These are written to `source_data/` rather than plotted.

In [ ]:
for q, df in fig1c_tracks.items():
    counts = df.leafcutter2_category.value_counts().to_dict()
    print(f'{q:>4}: {len(df):3d} SRSF4 junctions  {counts}')

### Fig. 1d — classification by usage quartile

Junctions split into quartiles by usage, then the share of each LeafCutter2
class. Frequently used junctions are mostly productive; rarely used ones are
mostly not.

In [ ]:
P.plot_fig1d(fig1d_data)
P.save_panel('fig1d', PLOTS_DIR)

pct = fig1d_data.pivot(index='quartile', columns='category', values='pct_junctions').round(1)
print(pct.to_string())

### Fig. 1e — GENCODE composition of each class

What GENCODE v46 calls the junctions that LeafCutter2 assigns to each class.

In [ ]:
top = (fig1e_data.groupby('leafcutter2_category')
       .apply(lambda g: g.nlargest(4, 'n_junctions')[['gencode_annotation', 'pct_of_class']])
       )
print(top.round(1).to_string())

### Fig. 1g — log2 fold change, unproductive vs productive

Cumulative distribution of per-junction log2 fold change across four
perturbations that each stabilise NMD substrates. Unproductive junctions shift
right in every one; productive junctions sit on zero.

In [ ]:
P.plot_fig1g(fig1g_data)
P.save_panel('fig1g', PLOTS_DIR)

for s in fig1g_data:
    if s['category'] != 'utr':
        print(f"{s['comparison_label']:<36} {s['category_label']:<13} "
              f"n = {s['n']:>7,}   median log2FC = {s['median_log2fc']:+.3f}")

### Fig. 1h — rules predicting NMD efficiency

Each rule splits unproductive junctions into the group it predicts is degraded
more efficiently and the group it predicts is degraded less, and compares their
naRNA-vs-polyA log2 fold change with a one-sided Mann-Whitney U test. The
productive column is the matched coding-junction control.

The 50-nt rule is stated as *further* than 50 nt from the last EJC, because that
is the direction the rule predicts: a PTC within 50 nt of the last junction
escapes NMD. Grouping it the other way inverts the sign.

In [ ]:
P.plot_fig1h(fig1h_data)
P.save_panel('fig1h', PLOTS_DIR)

cols = ['rule', 'class', 'n_low', 'n_high', 'mean_low', 'mean_high',
        'delta_log2fd', 'p_value']
print(fig1h_data[cols].to_string(index=False))